
# Fusion des données RPPS et agrégation de l'offre de soins par commune

Ce notebook fusionne la table personne-activité avec la table savoir-faire, puis agrège l'offre de soins au niveau communal.

**Objectif :**
- charger les tables RPPS préparées ;
- fusionner les informations sur les professionnels et leurs savoir-faire ;
- compter le nombre de professionnels distincts par commune ;
- produire une table d'offre de soins communale.


In [5]:
import pandas as pd

In [6]:

personne = pd.read_csv(
    "s3://mangane/Projet_PDS_2026/processed/personne_activite_clean.csv",
    dtype={"Identifiant PP": str}
)

savoirfaire = pd.read_csv(
    "s3://mangane/Projet_PDS_2026/processed/savoirfaire_clean.csv",
    dtype={"Identifiant PP": str}
)


/tmp/ipykernel_4611/521689486.py:1: DtypeWarning: Columns (10: Code postal (coord. structure)) have mixed types. Specify dtype option on import or set low_memory=False.
  personne = pd.read_csv(


In [7]:

table_finale = pd.merge(
    personne,
    savoirfaire,
    on="Identifiant PP",
    how="left"
)

print("Dimensions de la table finale :")
print(table_finale.shape)

print("\n10 premières lignes :")
print(table_finale.head(10).to_string())


Dimensions de la table finale :
(2395572, 17)

10 premières lignes :
  Identifiant PP Libellé profession_x Code mode exercice Libellé mode exercice Numéro FINESS site              Raison sociale site Code secteur d'activité     Libellé secteur d'activité Code commune (coord. structure) Libellé commune (coord. structure) Code postal (coord. structure)  Code Département (structure) Libellé profession_y Code type savoir-faire Libellé type savoir-faire Code savoir-faire           Libellé savoir-faire
0    10000001056              Médecin                  S               Salarié          970300026    CENTRE HOSPITALIER DE CAYENNE                    SA01  Etablissement Public de santé                           97302                            Cayenne                        97306.0                           NaN              Médecin                      S       Spécialité ordinale              SM20        Gynécologie-obstétrique
1    10000001452           Sage-Femme                NaN         

In [8]:

offre_commune = (
    table_finale
    .groupby(
        [
            "Code commune (coord. structure)",
            "Libellé commune (coord. structure)"
        ]
    )
    .agg(
        nb_professionnels=("Identifiant PP", "nunique")
    )
    .reset_index()
)

print("\nDimensions de la table par commune :")
print(offre_commune.shape)

print("\n10 premières communes :")
print(offre_commune.head(10).to_string())



Dimensions de la table par commune :
(17213, 3)

10 premières communes :
  Code commune (coord. structure) Libellé commune (coord. structure)  nb_professionnels
0                           00000                            Inconnu                  9
1                           01001            L'Abergement-Clémenciat                  1
2                           01002              L'Abergement-de-Varey                  1
3                           01004                  Ambérieu-en-Bugey                521
4                           01005                Ambérieux-en-Dombes                 29
5                           01007                           Ambronay                 28
6                           01010                          Anglefort                  4
7                           01014                             Arbent                 48
8                           01015                    Arboys en Bugey                  1
9                           01021             

In [9]:

offre_commune.to_csv(
    "s3://mangane/Projet_PDS_2026/processed/offre_soins_commune.csv",
    index=False
)

print("\nFichier enregistré : offre_soins_commune.csv")



Fichier enregistré : offre_soins_commune.csv
